In [354]:
import os
import re
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [355]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [356]:
db_fetched_data = """
오늘 체온 36.7도. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림. 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨자.
"""

In [357]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()


In [358]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 450  
}

extractor_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)

In [359]:
extract_prompt = f"""[Instruction] 
당신은 육아 기록 및 텍스트 분석 전문가입니다. 
주어진 [Data]의 텍스트를 줄바꿈과 문장부호(., !, ?) 기준으로 빠짐없이 분리하세요.
그 후, 원문 문장 순서를 그대로 유지하면서 아래의 [Output Format]에 맞춰 각 문장을 정밀 분석한 결과를 출력하세요.
수식어 없이 원문과 관련있는 내용들만 출력하세요.

각 문장 분석 시 아래 규칙을 철저히 따르세요:
1. 핵심명사: 문장의 주제가 되는 주요 고유명사나 일반명사 (한 문장에 여러 개가 존재하면 누락 없이 모두 작성 / 없으면 '없음')
2. 행동명사: 문장에서 일어나는 행위, 동작, 상태 변화를 나타내는 명사 (한 문장에 여러 개가 존재하면 누락 없이 모두 작성 / 없으면 '없음')
3. 수치+단위: 문장에 포함된 숫자와 단위의 조합 (예: 36.7도, 15분, 7번 / 없으면 '없음')
4. 예측 감정단어: 문맥을 파악하여 작성자(엄마)의 심리 상태나 감정을 나타내는 단어 1개 (예: 안도, 염려, 지침, 기특, 간절 등)

[Data]
{step1_insights}

[Output Format]
1. [첫 번째 문장 원문]
- 핵심명사: 단어1, 단어2, 단어3 (문장에 등장하는 모든 주요 명사를 쉼표로 나열)
- 행동명사: 단어1, 단어2 (문장에 등장하는 모든 행위 및 동작 명사를 쉼표로 나열)
- 수치+단위: 
- 예측 감정단어: 
- 육아 범주 분류 (식사, 수면, 배변, 신체 활동 중 선택) :

2. [두 번째 문장 원문]
- 핵심명사: 단어1, 단어2
- 행동명사: 단어1, 단어2
- 수치+단위: 
- 예측 감정단어: 
- 육아 범주 분류 (식사, 수면, 배변, 신체 활동 중 선택) :

[Answer]:"""


In [360]:
# 1단계 추출 모델 실행 및 결과 받아오기
extract_response = extractor_model.generate(prompt=extract_prompt)
results_list = extract_response.get('results', [])
first_result = next(iter(results_list)) if isinstance(results_list, list) else results_list
step2_keywords = first_result.get('generated_text', '').strip()

perfect_match_input = step2_keywords


In [361]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",
    GenParams.TEMPERATURE: 0.7,
    GenParams.TOP_P: 0.85,
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 1200
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)


In [362]:
diary_prompt = f"""너는 인스타그램에 감성 가득한 일기를 공유하며 아이의 성장을 기록하는 다정하고 따뜻한 대한민국 엄마이다.
제공된 [입력 데이터]에는 번호가 붙은 라벨 덩어리당 실제 육아 원문과 엄마의 속마음인 '예측 감정단어'가 정확히 하나씩 매칭되어 있다.
반드시 각 번호의 원문 사건 내용에 해당 번호의 '예측 감정단어'가 가진 심리 상태를 감성적으로 융합하여, 당시 엄마의 마음이 깊이 있게 묻어나는 다정한 일기를 번호당 정확히 1문장씩 완성해라.

[필수 제약 규칙]
1. 예측 감정 중심의 문장 재창조: 데이터에 적힌 안도, 염려, 기특, 간절 같은 감정 단어 자체를 문장에 직접 노출하지 마라. 그 감정이 뜻하는 엄마의 다정하고 따뜻한 속마음 뉘앙스(예: 안도란 마음이 놓이다, 염려란 걱정이 앞서다, 기특이란 얼마나 대견한지 모른다, 간절이란 마음속으로 간절히 기도하다 등)를 문장 전체의 서술과 어조에 자연스럽게 녹여내어 감성을 극대화해라. 단, 번역투 표현인 '잘 수가 있었어요' 대신 '푹 잠들어 주었네요'나 '잘 자주었네요'를 사용하고, '들썩들썩하게 해서' 대신 '들썩들썩하느라'를 사용하며, '바래요' 대신 표준어인 '바라요' 혹은 '기도해요'를 사용해라.
2. 번호 내용 독립 매칭: 제공된 데이터의 각 번호를 순서대로 처리하되, 해당 번호에 적힌 실제 사건 내용만 가지고 일기를 써라. 6번은 뒤집기 시도와 땀 사건에 기특한 감정을 쓰고, 7번은 밤 9시 막수 사건에 간절한 감정을 써야 하며 앞선 번호의 유모차나 배앓이 이야기를 중복해서 지어내지 마라. 번호당 정확히 1문장씩 총 7문장으로 끝내라.
3. 문장 종결 어미 단일화: 문장 끝에 했어요 했답니다 처럼 두 개의 종결 어미를 쉼표로 연결하여 중복 출력하지 마라. 한 문장에는 반드시 단 하나의 자연스러운 서술어 어미만 사용하여 문장을 깔끔하게 끝맺어라.
4. 원문 명사형 어미 완전 변형: 원문에 적힌 뀜, 직수함, 흘림 같은 명사형 표현을 그대로 복사하지 마라. 뀌어서 또는 뀌어대니, 직수했답니다, 흘렸네요와 같이 한국인 엄마가 쓰는 다정한 표준 서술어로 바꾸어 문장을 새로 작성해라.
5. 존댓말 어조 통일: 모든 문장은 인스타그램 독자에게 고백하는 듯한 높임말 존댓말 구어체로 일관되게 작성해야 한다. 문장 끝에 반말이나 독백형 낮춤말을 절대 쓰지 마라.
6. 불필요한 주어 반복 금지: 매 문장마다 아기는, 아이가 같은 주어를 기계적으로 반복하지 마라. 흐름상 자연스럽게 주어를 생략하여 가독성을 높여라.
7. 수치 기호 필수 노출: 제공된 메모의 수치 단위 기호(36.7도, 15분, 7번, 40분, 9시, 1번 등)는 문장 속에 기호 형태 그대로 명확하게 노출하여 사용해라.
8. 순번 기호 사용 금지: 문장 앞이나 뒤에 숫자가 적힌 순번 기호를 절대 사용하지 마라. 한 문장이 끝날 때마다 무조건 줄바꿈만 수행해라. 임의의 마무리 총평 문장을 절대 지어내지 마라.
9. 마지막 번호에 대응하는 일기 문장을 마친 직후, 다음 줄에 무조건 [END] 라고만 선명하게 출력하고 생성을 즉시 종결해라.

[입력 데이터]
{perfect_match_input}

[Diary]:"""


In [363]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) else writer_results
raw_diary = first_writer_result.get('generated_text', '').strip()

In [364]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

# 2. 텍스트 정제 (한자 및 특수문자 제거)
cleaned_diary = re.sub(r'[\u4e00-\u9fff]', '', raw_diary) 
cleaned_diary = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?]', '', cleaned_diary) # 'ml', '도' 기호 보존용
cleaned_diary = re.sub(r'^\d+[\.\s\-~)]+', '', cleaned_diary, flags=re.MULTILINE) # 시작 넘버링 제거

# 3. 모델이 출력한 줄바꿈(\n)을 우선 신뢰하여 분리
diary_lines = [line.strip() for line in cleaned_diary.split('\n') if line.strip()]

# 4. 각 라인별 재정제
full_print_lines = []
for line in diary_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+', '', line).strip()
    if line:
        full_print_lines.append(line)

# 5. 안전한 바이트 단위 축소 알고리즘 (한글 깨짐 방지)
def truncate_by_bytes(text, max_bytes=230):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    
    # 안전하게 지정된 바이트만큼 자른 후, 깨진 바이트 무시하고 디코딩
    truncated = text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore')
    return truncated.strip() + "..."

final_lines = []
for line in full_print_lines:
    # UTF-8 기준 바이트 수 체크
    line_bytes = line.encode('utf-8')
    
    if len(line_bytes) > 230:
        # 안전하게 227바이트까지 자른 후 깨진 멀티바이트 찌꺼기는 무시하고 디코딩
        line = line_bytes[:227].decode('utf-8', errors='ignore').strip() + "..."
        
    final_lines.append(line)

# 최종 다이어리 텍스트 병합 결과물
final_diary = "\n".join(final_lines)

In [365]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 오늘 체온 36.7도.
2. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜.
3. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함.
4. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌.
5. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦.
6. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림.
7. 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨자.


In [366]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 오늘 체온 36.7도.
- 핵심명사: 체온
- 행동명사: 없음
- 수치+단위: 36.7도
- 예측 감정단어: 안도
- 육아 범주 분류: 신체 활동

2. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜.
- 핵심명사: 대변, 방귀
- 행동명사: 없음
- 수치+단위: 없음
- 예측 감정단어: 염려
- 육아 범주 분류: 배변

3. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함.
- 핵심명사: 완모 수유, 양쪽, 직수
- 행동명사: 수유, 직수
- 수치+단위: 15분, 7번
- 예측 감정단어: 안도
- 육아 범주 분류: 식사

4. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌.
- 핵심명사: 배앓이
- 행동명사: 먹다, 짜증, 트림
- 수치+단위: 없음
- 예측 감정단어: 염려
- 육아 범주 분류: 식사

5. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦.
- 핵심명사: 낮잠, 유모차, 동네
- 행동명사: 태우다, 도는
- 수치+단위: 40분
- 예측 감정단어: 안도
- 육아 범주 분류: 수면

6. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림.
- 핵심명사: 뒤집기,


In [367]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 오늘 36.7도라는 체온을 측정했을 때, 마침내 건강한 모습을 보니 마음이 놓이네요.
[2번 일기]: 대변은 보지 않고 방귀만 엄청 뽕뽕 뀌어서 오늘 배변 상태가 어떨지 걱정이 앞섭니다.
[3번 일기]: 오늘 양쪽으로 합쳐서 15분씩 총 7번 직수했더니 완모 수유가 잘 된 것 같아 마음이 놓여요.
[4번 일기]: 먹다가 자꾸 젖을 빼고 짜증을 내서 배앓이가 아닐까 싶어 트림을 열심히 시켜줬지만 그래도 마음이 놓이질 않네요.
[5번 일기]: 유모차에 태워서 동네를 한 바퀴 돌아보니 40분 동안 푹 잘 주었네요.
[6번 일기]: 뒤집기 시도를 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘렸지만 얼마나 대견한지 모릅니다.
[7번 일기]: 밤 9시에 막 수유를 끝내니 마음속으로 건강하고 행복하게 자라고 기도해요.

-> 최종 결과물 총 문장 수: 7줄
